# Research on Right Augmented Chow Function $F=G$

This notebook explores the definitions and properties of the right augmented Chow function ($F$) and the left augmented Chow function ($G$).

## Definitions

*   **Chow function** ($\underline{H}$): The polynomial $H_M(x)$ associated with the Chow ring of a matroid.
*   **Right augmented Chow function** ($F$): Defined as $F = \underline{H} \cdot f^{\text{rev}}$.
*   **Left augmented Chow function** ($G$): Defined as $G = g^{\text{rev}} \cdot \underline{H}$.

The convolution $A \cdot B$ is defined as $(A \cdot B)(M) = \sum_{F \in L(M)} A(M^F) B(M_F)$.

In [2]:
import sys
import os
import sage.all
from sage.matroids.advanced import *

# Setup paths to import custom functions
sys.path.append(os.path.abspath('..'))
from custom_matroid_functions.chow_polynomials import chow_polynomial, characteristic_polynomial

P.<x> = LaurentPolynomialRing(ZZ)

def convolve(a, b, M):
    sum = P(0)
    L = M.lattice_of_flats()
    for F in L:
        sum += a(M.restriction(F)) * b(M.contract(F))
    return sum

def chow(M):
    return chow_polynomial(M)

def kl_rev(M):
    # Value of f_rev: x^rk * P(M)(x^-1)
    p = M.lattice_of_flats().kazhdan_lusztig_polynomial()(x)
    return p(x**-1) * x**M.rank()

def g_rev(M):
    # Left KLS rev function. For matroids, g=f, so g_rev = f_rev
    return kl_rev(M)

def right_aug_chow(M):
    # F = H * f_rev
    return convolve(chow, kl_rev, M)

def left_aug_chow(M):
    # G = g_rev * H
    return convolve(g_rev, chow, M)

def check_symmetry(p, rank):
    if p == 0: return True
    return p == p(x**-1) * x**rank

def standard_aug_chow(M):
    # A(M) = sum_{F} x^rk(F) * H(M/F)
    sum = P(0)
    L = M.lattice_of_flats()
    for F in L:
        sum += x**M.rank(F) * chow_polynomial(M.contract(F))(x)
    return sum

## Verification of $F=G$ Conjecture

We test if $F(M) = G(M)$ and if they both equal the standard augmented Chow polynomial $A(M) = \sum x^{\text{rk}
 F} H(M_F)$.

In [3]:
matroids_to_test = [
    matroids.CompleteGraphic(3),
    matroids.CompleteGraphic(4),
    matroids.Uniform(2, 3),
    matroids.Uniform(3, 4)
]

for M in matroids_to_test:
    print(f"Testing {M.name() if hasattr(M, 'name') else str(M)}...")
    f = right_aug_chow(M)
    g = left_aug_chow(M)
    a = standard_aug_chow(M)
    
    print(f"  F: {f}")
    print(f"  G: {g}")
    print(f"  A: {a}")
    
    sym_f = check_symmetry(f, M.rank())
    sym_g = check_symmetry(g, M.rank())
    
    print(f"  F symmetric: {sym_f}")
    print(f"  G symmetric: {sym_g}")
    
    if f == g:
        print("  F == G: OK")
    else:
        print(f"  F != G: Mismatch! Difference: {f - g}")
    
    if f == a:
        print("  F == A: OK")
    else:
        print(f"  F != A: Mismatch!")
    print("-" * 20)

Testing M(K3): Graphic matroid of rank 2 on 3 elements...


AttributeError: 'GraphicMatroid' object has no attribute 'restriction'

## Map $\Psi$ on Kazhdan-Lusztig Basis

Verification of $\Psi(s) = x^{\rk s} \underline{H}^{\text{rev}}(M^s)(x^{-2})$ and its effect on the KL basis.

In [ ]:
def verify_psi_on_kl_basis(M):
    L = M.lattice_of_flats()
    for t in L:
        m_t = M.restriction(t)
        image_psi = P(0)
        for s in L:
            if M.closure(s) == s and s.issubset(t):
                m_s = M.restriction(s)
                h_s = chow_polynomial(m_s)
                psi_s = x**M.rank(s) * h_s(x**-2)
                kl_st = M.restriction(t).lattice_of_flats().kazhdan_lusztig_polynomial(M.restriction(s))(x)
                image_psi += psi_s * kl_st
        
        f_mt = right_aug_chow(m_t)
        target = x**m_t.rank() * f_mt(x**-2)
        
        if image_psi != target:
             print(f"Mismatch for flat {t}")
             print(f"  Psi: {image_psi}")
             print(f"  Target: {target}")
    print("Map Verification OK")

M = matroids.CompleteGraphic(3)
verify_psi_on_kl_basis(M)